# Proposal Indicator Analysis

This notebook parses a folder of PDF and Word proposal documents, identifies candidate indicator sections, extracts structured M&E indicators with an LLM, matches them against a reerence library, and produces validation outputs and charts.


In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
import pandas as pd
import matplotlib.pyplot as plt
import extraction_utils

ROOT = Path('.')
PROPOSALS_DIR = ROOT / 'proposals'
OUTPUTS_DIR = ROOT / 'outputs'
FIGURES_DIR = OUTPUTS_DIR / 'figures'
OUTPUTS_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)

# Reference library: built from the ZOA IndiKit export (see "Build the reference library" cell).
ZOA_INDICATORS_PATH = ROOT / 'MEAL_Proposal-main' / 'MEAL_Proposal-main' / 'notebooks' / 'data' / 'zoa_indikit_net' / 'indicators.csv'
REFERENCE_LIBRARY_PATH = ROOT / 'reference_library.csv'
ENV_PATH = ROOT / '.env'
load_dotenv(dotenv_path=ENV_PATH)   # loads OPENAI_API_KEY / AZURE_OPENAI_* from .env

# ---- LLM provider / model -------------------------------------------------------------
# Exactly one option should be active. Uncomment the block you want and comment out the others.

# Option A (active): OpenAI gpt-4o-mini, as specified in the plan. Requires OPENAI_API_KEY in .env.
# All cached extraction responses in outputs/api_cache were produced with this model, so re-runs are free.
LLM_PROVIDER = 'openai'
MODEL_NAME = 'gpt-4o-mini'

# Option B: local Llama 3.2 through Ollama. Install Ollama (https://ollama.com), then: ollama pull llama3.2
# LLM_PROVIDER = 'ollama'
# MODEL_NAME = 'llama3.2'            # e.g. 'llama3.2' (3B) or 'llama3.2:1b'
# OLLAMA_BASE_URL = 'http://localhost:11434/v1'

# Option C: Azure OpenAI. Requires AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_API_KEY in .env
#           (optionally AZURE_OPENAI_API_VERSION). MODEL_NAME is your *deployment name*, not the base model.
# LLM_PROVIDER = 'azure'
# MODEL_NAME = 'gpt-4o-mini'       # <- replace with the deployment name from the Azure portal
# AZURE_OPENAI_ENDPOINT = os.getenv('AZURE_OPENAI_ENDPOINT', 'https://<your-resource>.openai.azure.com/')
# AZURE_OPENAI_API_VERSION = os.getenv('AZURE_OPENAI_API_VERSION', '2024-10-21')
# ----------------------------------------------------------------------------------------

FORCE_REFRESH = False           # re-call the LLM even when a cached response exists
EMBEDDING_MODEL = 'all-MiniLM-L6-v2'
MIN_CHUNK_SCORE = 0.30
KEEP_NEIGHBORS = 1
LINK_TOP_K = 5                  # number of ZOA candidates shown to the LLM per proposal indicator (Phase 4)

extraction_utils.FORCE_REFRESH = FORCE_REFRESH  # the cache helpers read the module-level flag
llm_config = extraction_utils.configure_llm(
    provider=LLM_PROVIDER,
    model_name=MODEL_NAME,
    base_url=globals().get('OLLAMA_BASE_URL'),
    azure_endpoint=globals().get('AZURE_OPENAI_ENDPOINT'),
    azure_api_version=globals().get('AZURE_OPENAI_API_VERSION'),
)

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')   # kept for reference; provider readiness is checked below

LLM_AVAILABLE, llm_status = extraction_utils.llm_available()
print(f'LLM provider: {llm_config["provider"]} | model: {llm_config["model"]} | endpoint: {llm_config["base_url"]}')
print(f'LLM ready: {LLM_AVAILABLE} ({llm_status})')

proposal_paths = sorted(PROPOSALS_DIR.rglob('*.pdf')) + sorted(PROPOSALS_DIR.rglob('*.docx'))
proposal_paths = [p for p in proposal_paths if p.is_file()]
print(f'Found {len(proposal_paths)} proposal documents')


In [ ]:
from extraction_utils import extract_chunks_from_file, filter_indicator_chunks, LLMClient, deduplicate_indicators, match_indicators_to_library

all_chunks = []
parse_errors = []
for path in proposal_paths:
    chunks, errors = extract_chunks_from_file(str(path))
    all_chunks.extend(chunks)
    parse_errors.extend(errors)

print(f'Parsed {len(all_chunks)} chunks from {len(proposal_paths)} documents')
if parse_errors:
    print(f'Encountered {len(parse_errors)} parse issues')


## Section targeting and candidate selection
Score all chunks for indicator likelihood, keep the highest-scoring chunks plus neighbors, and record coverage for each document.


In [ ]:
chunks_by_doc = {}
coverage_records = []
candidate_chunks = []
for path in proposal_paths:
    doc_id = path.name
    doc_chunks = [chunk for chunk in all_chunks if chunk.doc_id == doc_id]
    selected, summary = filter_indicator_chunks(doc_chunks, min_score=MIN_CHUNK_SCORE, keep_neighbors=KEEP_NEIGHBORS)
    chunks_by_doc[doc_id] = {
        'all_chunks': doc_chunks,
        'selected_chunks': selected,
        'summary': summary,
    }
    coverage_records.append({
        'doc_id': doc_id,
        'total_chunks': summary['total_chunks'],
        'selected_chunks': summary['selected_chunks'],
        'coverage_fraction': summary['coverage_fraction'],
    })
    candidate_chunks.extend(selected)
coverage_df = pd.DataFrame(coverage_records)
print(f'Selected {len(candidate_chunks)} candidate chunks across {len(chunks_by_doc)} documents')
coverage_df.head()


## LLM extraction
Extract structured indicators from the candidate chunks with the model chosen in the config cell. Responses are cached per chunk **and per model** under `outputs/api_cache`, so switching models never reuses another model's answers, and re-running with the same model is free.

If the LLM server is not reachable, this cell falls back to **cache-only mode**: it rebuilds the dataset from the cached responses and reports how many chunks had no cached answer. The save cells further down never overwrite an existing non-empty output file with an empty one.


In [ ]:
extracted_indicators = []
llm_failures = []
cache_misses = 0
client = LLMClient(model_name=MODEL_NAME)
if not LLM_AVAILABLE:
    print(f'LLM not available ({llm_status}).')
    print(f'Rebuilding from cached {MODEL_NAME} responses only; chunks without a cached response are counted as misses and skipped.')
for n, chunk in enumerate(candidate_chunks, 1):
    try:
        chunk_indicators = client.extract_indicators_from_chunk(chunk, prompt_version='v1', cache_only=not LLM_AVAILABLE)
        extracted_indicators.extend(chunk_indicators)
    except extraction_utils.CacheMiss:
        cache_misses += 1
    except Exception as exc:
        llm_failures.append({'doc_id': chunk.doc_id, 'source_file': chunk.source_file, 'error': str(exc)})
    if n % 500 == 0:
        print(f'  {n} / {len(candidate_chunks)} chunks processed, {len(extracted_indicators)} indicators so far')
print(f'Extracted {len(extracted_indicators)} indicators from {len(candidate_chunks)} chunks with {MODEL_NAME}')
if cache_misses:
    print(f'WARNING: {cache_misses} chunks had no cached response and were skipped. Start the LLM and re-run this cell to extract them.')
if llm_failures:
    print(f'Encountered {len(llm_failures)} extraction failures. First error: {llm_failures[0]["error"][:200]}')


## Deduplication and assembly
Deduplicate extracted indicators within each proposal and assemble the final indicator dataset.


In [ ]:
deduped_indicators = []
for doc_id in sorted({ind.doc_id for ind in extracted_indicators}):
    items = [ind for ind in extracted_indicators if ind.doc_id == doc_id]
    keep, merged = deduplicate_indicators(items)
    deduped_indicators.extend(keep)
indicators_df = pd.DataFrame([ind.model_dump() for ind in deduped_indicators])
if indicators_df.empty:
    indicators_df = pd.DataFrame(columns=['indicator_text','level','baseline','target','means_of_verification','sector','disaggregation','unit_or_type','source_confidence','doc_id','chunk_ref'])
print('Assembled indicators dataframe with', len(indicators_df), 'rows')
indicators_df.head()


## Validation and outputs
Run basic QA checks on the extracted dataset and persist results to CSV and Excel.


In [ ]:
qa_checks = []
qa_checks.append({'check': 'missing_indicator_text', 'count': int(indicators_df['indicator_text'].isna().sum() if not indicators_df.empty else 0)})
qa_checks.append({'check': 'missing_level', 'count': int(indicators_df['level'].isna().sum() if not indicators_df.empty else 0)})
qa_checks.append({'check': 'duplicate_rows', 'count': int(indicators_df.duplicated(subset=['indicator_text', 'doc_id']).sum() if not indicators_df.empty else 0)})
qa_checks.append({'check': 'proposals_with_zero_indicators', 'count': int((coverage_df['selected_chunks'] == 0).sum())})
qa_df = pd.DataFrame(qa_checks)
qa_df


In [ ]:
indicators_csv = OUTPUTS_DIR / 'indicators_extracted.csv'
indicators_xlsx = OUTPUTS_DIR / 'indicators_extracted.xlsx'
if indicators_df.empty and indicators_csv.exists() and len(pd.read_csv(indicators_csv)) > 0:
    print(f'No indicators extracted in this run; keeping the existing {indicators_csv} untouched.')
else:
    indicators_df.to_csv(indicators_csv, index=False)
    with pd.ExcelWriter(indicators_xlsx, engine='openpyxl') as writer:
        for doc_id, group in indicators_df.groupby('doc_id'):
            safe_name = str(doc_id)[:31].replace('/', '_').replace('\\', '_')
            group.to_excel(writer, sheet_name=safe_name or 'doc', index=False)
        indicators_df.to_excel(writer, sheet_name='Combined', index=False)
    print('Saved', indicators_csv, 'and', indicators_xlsx)


## Matching against the reference library (legacy matching v1, deprecated)

`reference_library.csv` is **generated** by the metadata module from the merged IndiKit export + site data (344 export IDs + 7 site-only PIP indicators):

```
python -m pipeline.run build-metadata
```

The cell below only loads that file; if it is missing the loader raises `FileNotFoundError` (the earlier silent 3-row dummy library was removed). `match_indicators_to_library` is the retired fuzzy/embedding matcher and now emits a `DeprecationWarning`; the Phase 3 LLM agent (`pipeline/agent/`) replaces it. See `docs/plan.md`.


In [ ]:
# Load the reference library generated by the metadata module (python -m pipeline.run build-metadata)
from extraction_utils import load_reference_library

reference_df, _ = load_reference_library(str(REFERENCE_LIBRARY_PATH))   # raises FileNotFoundError if not built
print(f'Loaded {REFERENCE_LIBRARY_PATH} with {len(reference_df)} IndiKit indicators across {reference_df["sector"].nunique()} sectors')
reference_df.head()


In [ ]:
import warnings
with warnings.catch_warnings():
    warnings.simplefilter('ignore', DeprecationWarning)   # matching v1 is deprecated (see docs/plan.md Section 5)
    matches_df, _ = match_indicators_to_library(deduped_indicators, str(REFERENCE_LIBRARY_PATH), embedding_model=EMBEDDING_MODEL)
matches_csv = OUTPUTS_DIR / 'indicator_matches.csv'
if matches_df.empty and matches_csv.exists() and len(pd.read_csv(matches_csv)) > 0:
    print(f'No indicators to match in this run; keeping the existing {matches_csv} untouched.')
else:
    matches_df.to_csv(matches_csv, index=False)
    print('Saved', matches_csv, 'with', len(matches_df), 'match records')

if not matches_df.empty:
    if EMBEDDING_MODEL and (matches_df['embedding_score'] == 0).all():
        print('WARNING: all embedding scores are 0. The embedding step failed and matching fell back to lexical only. Check that fastembed is installed (pip install fastembed) and the model can be downloaded.')
    print('\nMatch status vs ZOA reference library:')
    print(matches_df['match_status'].value_counts().to_string())
    covered = matches_df['match_status'].isin(['exact', 'close', 'related']).sum()
    print(f'\nCovered by library (exact/close/related): {covered} of {len(matches_df)}; novel candidates for library expansion: {len(matches_df) - covered}')
matches_df.head()


## Statistics and visualizations
Generate summary charts for indicator counts, metadata completeness, and match status.


In [ ]:
def save_fig(fig, name):
    path = FIGURES_DIR / name
    fig.savefig(path, bbox_inches='tight')
    return path

if not indicators_df.empty:
    counts = indicators_df['doc_id'].value_counts().sort_values(ascending=False)
    fig, ax = plt.subplots(figsize=(10, 6))
    counts.plot(kind='bar', ax=ax)
    ax.set_title('Indicators per proposal')
    ax.set_ylabel('Count')
    ax.set_xlabel('Proposal')
    save_fig(fig, 'indicators_per_proposal.png')
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(8, 5))
    indicators_df['level'].fillna('unclear').value_counts().plot(kind='bar', ax=ax)
    ax.set_title('Indicator distribution by level')
    ax.set_ylabel('Count')
    save_fig(fig, 'indicator_level_distribution.png')
    plt.close(fig)

    sector_plot = indicators_df['sector'].fillna('unknown').value_counts()
    fig, ax = plt.subplots(figsize=(10, 5))
    sector_plot.plot(kind='bar', ax=ax)
    ax.set_title('Indicator distribution by sector')
    ax.set_ylabel('Count')
    save_fig(fig, 'indicator_sector_distribution.png')
    plt.close(fig)

    completeness = pd.DataFrame({
        'baseline': indicators_df['baseline'].notna() & (indicators_df['baseline'].astype(str) != ''),
        'target': indicators_df['target'].notna() & (indicators_df['target'].astype(str) != ''),
        'means_of_verification': indicators_df['means_of_verification'].notna() & (indicators_df['means_of_verification'].astype(str) != ''),
    }).mean().sort_values(ascending=True)
    fig, ax = plt.subplots(figsize=(8, 4))
    completeness.plot(kind='barh', ax=ax)
    ax.set_title('Metadata completeness share')
    save_fig(fig, 'metadata_completeness.png')
    plt.close(fig)

    if not matches_df.empty and 'match_status' in matches_df.columns:
        fig, ax = plt.subplots(figsize=(6, 6))
        matches_df['match_status'].value_counts().plot(kind='pie', autopct='%1.1f%%', ax=ax)
        ax.set_ylabel('')
        ax.set_title('Match status distribution vs reference library')
        save_fig(fig, 'match_status_distribution.png')
        plt.close(fig)

    fields = ['level', 'baseline', 'target', 'means_of_verification', 'sector', 'disaggregation', 'unit_or_type']
    heatmap_data = indicators_df[fields].notna().astype(int)
    if not heatmap_data.empty:
        fig, ax = plt.subplots(figsize=(10, max(4, len(heatmap_data) * 0.15)))
        ax.imshow(heatmap_data.values.T, aspect='auto', cmap='Greens')
        ax.set_yticks(range(len(fields)))
        ax.set_yticklabels(fields)
        ax.set_xticks(range(min(len(heatmap_data), 20)))
        ax.set_xticklabels(heatmap_data.index[:20].tolist(), rotation=90)
        ax.set_title('Indicator metadata completeness heatmap (first 20 rows)')
        save_fig(fig, 'completeness_heatmap.png')
        plt.close(fig)
else:
    print('No extracted indicators available to visualize yet.')


## Next steps: the ESGReveal pipeline

The earlier "Phase 4/5" linking cells (duplicated, `head(30)`, `client.chat` bug) were removed. The project now follows the three-module method of ESGReveal (Zou et al., 2023), see `docs/plan.md`:

- **Metadata module** (Phase 1, done): `python -m pipeline.run build-metadata` writes `pipeline/metadata/indikit_metadata.jsonl`, `reference_library.csv` and `outputs/metadata/build_report.md`.
- **Preprocessing module** (Phase 2): `python -m pipeline.run build-kb`.
- **LLM agent + evaluation** (Phase 3): `python -m pipeline.run query`, `run-experiments`; gold set under `pipeline/eval/gold/` (`validate-gold`).

The cell below inspects the metadata library.


In [ ]:
# IndiKit metadata library (one record per unique indicator ID)
from pipeline import config
from pipeline.metadata.build_metadata import read_jsonl

records = read_jsonl(config.METADATA_JSONL)
meta_df = pd.DataFrame([
    {'indikit_id': r.indikit_id, 'name': r.name, 'sector': r.primary_sector, 'sub_sector': r.primary_sub_sector,
     'levels': ' | '.join(r.levels) or '(missing)', 'quantity': r.quantity.value, 'on_site': r.on_site, 'in_export': r.in_export,
     'n_search_terms': len(r.search_terms), 'has_knowledge': bool(r.knowledge)}
    for r in records
])
print(f'{len(meta_df)} indicators; on site: {meta_df.on_site.sum()}, export-only: {(meta_df.in_export & ~meta_df.on_site).sum()}, site-only: {(meta_df.on_site & ~meta_df.in_export).sum()}')
print('\nPer sector:')
print(meta_df['sector'].value_counts().to_string())
print('\nPer level combination:')
print(meta_df['levels'].value_counts().to_string())
meta_df.head()
